# MultiMedAI — GPU training & eval on FREE Colab

Your laptop is AMD/Windows/CPU (no usable PyTorch GPU). This notebook uses a **free Colab T4 GPU** ($0) to TRAIN better artifacts, which you then **download and run locally on CPU**.

It improves the two CPU-measured baselines:

| Component | CPU baseline (measured) | This notebook |
|-----------|------------------------|---------------|
| Retrieval | R@1 8.4% / R@5 21.6% / R@10 29.2% | improved captions + full bank |
| VQA | not measured on CPU | trained head (image+question) |

(Synthesis LoRA fine-tune is in the separate `finetune_sd_lora_colab.ipynb`.)

**Every number printed is real.** Free-GPU + subset = honest demonstration, not a full-scale benchmark.

### How to run
1. Colab (https://colab.research.google.com) → Upload this notebook.
2. `Runtime → Change runtime type → T4 GPU`.
3. `Runtime → Run all`. ~15–25 min.
4. Download the artifacts at the end into your repo's `weights/`.

## 1. GPU check + install

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install open_clip_torch==2.32.0 datasets==3.0.1 transformers==4.46.0 2>/dev/null
print('installed')

## 2. Load PathVQA + build IMPROVED per-image captions

Key retrieval improvement: instead of raw `"question answer"`, aggregate each image's open-ended answers into a descriptive caption: `"histopathology, H&E. showing: {answer1}; {answer2}; ..."`. Better text → better text→image alignment.

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import hashlib

def img_hash(img):
    return hashlib.md5(img.convert('L').resize((64,64)).tobytes()).hexdigest()

def build_unique(split, limit=None):
    ds = load_dataset('flaviagiammarino/path-vqa', split=split)
    images, ans_by_img, order = {}, defaultdict(list), []
    for ex in ds:
        h = img_hash(ex['image'])
        if h not in images:
            images[h] = ex['image']; order.append(h)
        a = ex['answer'].strip().lower()
        if a not in ('yes','no') and 2 < len(a) < 60:
            ans_by_img[h].append(a)
        if limit and len(order) >= limit:
            break
    caps = {}
    for h in order:
        uniq = list(dict.fromkeys(ans_by_img[h]))[:6]
        caps[h] = 'histopathology image, H&E stain. showing: ' + ('; '.join(uniq) if uniq else 'tissue')
    return order, images, caps

val_order, val_imgs, val_caps = build_unique('validation', limit=500)
print('unique val images:', len(val_order))
print('example caption:', val_caps[val_order[0]])

## 3. BiomedCLIP on GPU — retrieval Recall@k with improved captions

In [ ]:
import open_clip, torch
name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
model, preprocess = open_clip.create_model_from_pretrained(name)
tokenizer = open_clip.get_tokenizer(name)
model = model.eval().cuda()

@torch.no_grad()
def enc_imgs(pils, B=64):
    out=[]
    for i in range(0,len(pils),B):
        x=torch.stack([preprocess(p.convert('RGB')) for p in pils[i:i+B]]).cuda()
        f=model.encode_image(x); f=f/f.norm(dim=-1,keepdim=True); out.append(f.cpu())
    return torch.cat(out)

@torch.no_grad()
def enc_txts(txts, B=64):
    out=[]
    for i in range(0,len(txts),B):
        t=tokenizer(txts[i:i+B]).cuda()
        f=model.encode_text(t); f=f/f.norm(dim=-1,keepdim=True); out.append(f.cpu())
    return torch.cat(out)

imgs=[val_imgs[h] for h in val_order]; caps=[val_caps[h] for h in val_order]
ie=enc_imgs(imgs); te=enc_txts(caps)
sims=te@ie.T; ranks=sims.argsort(1,descending=True); gt=torch.arange(len(caps)).unsqueeze(1)
for k in (1,5,10):
    r=((ranks[:,:k]==gt).any(1).float().mean().item())*100
    print(f'Recall@{k:<2d} = {r:5.1f}%   (CPU baseline: ' + {1:"8.4%",5:"21.6%",10:"29.2%"}[k] + ')')

## 4. VQA — train a head on BiomedCLIP (image + question) embeddings

Input = concat(image_emb, question_text_emb) = 1024-d (both from BiomedCLIP's shared space). Head = small MLP → closed vocab of top-N answers. Encoder frozen; only the head trains. GPU makes many epochs cheap.

In [ ]:
from datasets import load_dataset
from collections import Counter
import torch

TOPN = 300
tr = load_dataset('flaviagiammarino/path-vqa', split='train')
va = load_dataset('flaviagiammarino/path-vqa', split='test')

freq = Counter(ex['answer'].strip().lower() for ex in tr)
vocab = [a for a,_ in freq.most_common(TOPN)]
a2i = {a:i for i,a in enumerate(vocab)}
print('vocab size:', len(vocab), '| top:', vocab[:8])

def featurize(ds, limit=None):
    imgs, qs, ys = [], [], []
    for ex in ds:
        a = ex['answer'].strip().lower()
        if a not in a2i:
            continue
        imgs.append(ex['image']); qs.append(ex['question']); ys.append(a2i[a])
        if limit and len(ys) >= limit:
            break
    ie = enc_imgs(imgs); qe = enc_txts(qs)
    X = torch.cat([ie, qe], dim=1)
    return X, torch.tensor(ys), qs

Xtr, ytr, _   = featurize(tr, limit=8000)
Xva, yva, qva = featurize(va, limit=3000)
print('train', Xtr.shape, '| val', Xva.shape)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(42)
head = nn.Sequential(nn.Linear(1024,512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512,len(vocab))).cuda()
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()
Xtr_g, ytr_g = Xtr.cuda(), ytr.cuda(); Xva_g = Xva.cuda()

for epoch in range(60):
    head.train(); opt.zero_grad()
    loss = lossf(head(Xtr_g), ytr_g); loss.backward(); opt.step()
    if (epoch+1) % 10 == 0:
        head.eval()
        with torch.no_grad():
            acc = (head(Xva_g).argmax(1).cpu() == yva).float().mean().item()*100
        print(f'epoch {epoch+1:2d}  loss {loss.item():.3f}  val_acc {acc:.2f}%')

## 5. VQA accuracy — overall + yes/no vs open-ended split (the honest breakdown)

In [ ]:
head.eval()
with torch.no_grad():
    pred = head(Xva_g).argmax(1).cpu()
correct = (pred == yva)
yn_mask = torch.tensor([vocab[y] in ('yes','no') for y in yva.tolist()])
overall = correct.float().mean().item()*100
yn = correct[yn_mask].float().mean().item()*100 if yn_mask.any() else float('nan')
oe = correct[~yn_mask].float().mean().item()*100 if (~yn_mask).any() else float('nan')
print('='*50)
print(f'REAL VQA accuracy (val, n={len(yva)})')
print(f'  overall    = {overall:.2f}%')
print(f'  yes/no     = {yn:.2f}%   (n={int(yn_mask.sum())}, ~50% chance)')
print(f'  open-ended = {oe:.2f}%   (n={int((~yn_mask).sum())}, {len(vocab)}-way)')
print('='*50)

## 6. Export artifacts → download into your repo's `weights/`

In [ ]:
import torch, json
torch.save(head.state_dict(), 'vqa_head.pt')
with open('vqa_vocab.json','w') as f: json.dump(vocab, f)
print('saved vqa_head.pt + vqa_vocab.json')
try:
    from google.colab import files
    files.download('vqa_head.pt'); files.download('vqa_vocab.json')
except Exception:
    print('Not on Colab — download the two files manually.')

## Done — wire-up locally

1. Put `vqa_head.pt` + `vqa_vocab.json` in your repo at `weights/vqa/`.
2. Put the LoRA `multimedai_lora.safetensors` (from the other notebook) in `weights/lora/`.
3. The local Streamlit app loads these for CPU inference — BiomedCLIP embeds the image+question on CPU, the tiny head predicts instantly.
4. Record the real numbers (retrieval Recall@k, VQA overall/yes-no/open-ended) in `DEFENSE.md`, alongside the CPU baselines, to show the GPU improvement.